## PEP-DQN 算法实现
### 创新点
1.  **优化动作参数选择空间**：通过引入确定性函数来映射状态和离散动作到连续参数，避免了对连续参数的穷举搜索，提高了效率。
2.  **引入优先回放缓冲区**：使智能体能够更关注那些“学习价值”高的样本，从而加速收敛。
3.  **利用专家样本**：在训练初期引入高质量的“专家”决策样本，解决冷启动问题，引导模型快速学习有效策略。

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
from collections import deque, namedtuple
import math
from typing import List, Tuple
import pandas as pd

# 设置随机种子以确保结果可复现
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


### 第一部分：构建环境与任务模型

- **任务 (Task)**: 由位置 $(X, Y)$、优先级 $P$、基础奖励 $R$ 和截止时间 $t_K$ 等描述。
一个关键概念是任务完成率 $D_{T_k}$ (公式1)
它是一个衰减因子，表示任务完成时间越接近截止日期，收益越低。
$$
D_{T_{k}} =
\begin{cases}
    1 / (1 + e^{-\epsilon(t_K - t_i K)}), & \text{if } t_K \geq t_i K \\
    0, & \text{if } t_K < t_i K
\end{cases}
$$

- **工人 (Worker)**: 由位置 $(X, Y)$、单位距离报酬 $B_{W_n}$、能力 $C_{W_n}$ 和速度 $V_{W_n}$ 描述。工人的成本是其移动距离和报酬的乘积 (公式3)，而其收益则与其能力、任务属性和完成率相关 (公式4)。
- **无人机 (UAV)**: 与工人类似，由位置、单位距离能耗 $B_{U_m}$、能力 $C_{U_m}$ 和速度 $V_{U_m}$ 描述。其成本是能耗 (公式7)，收益计算方式与工人类似 (公式8)。

In [2]:
class Task:
    """任务模型"""
    def __init__(self, task_id: int, x: float, y: float, priority: float,
                 base_reward: float, deadline: float):
        self.task_id = task_id
        self.x = x  # X 坐标
        self.y = y  # Y 坐标
        self.priority = priority  # 任务优先级 P_T_k
        self.base_reward = base_reward  # 基础奖励 R_T_k
        self.deadline = deadline  # 截止时间 t_T_k
        self.completed = False
        self.completion_time = None

    def get_completion_rate(self, completion_time: float, epsilon: float = 1.0) -> float:
        """公式(1)计算任务完成率 D_T_k"""
        if completion_time <= self.deadline:
            decay_factor = 1 / (1 + math.exp(-epsilon * (self.deadline - completion_time)))
            return decay_factor
        else:
            return 0.0

class Worker:
    """工人模型"""
    def __init__(self, worker_id: int, x: float, y: float, compensation: float,
                 competence: float, speed: float):
        self.worker_id = worker_id
        self.x = x  # X 坐标
        self.y = y  # Y 坐标
        self.compensation = compensation  # B_W_n (每公里报酬)
        self.competence = competence  # C_W_n (0.7-0.9)
        self.speed = speed  # V_W_n (4-5)
        self.available = True

    def calculate_distance(self, task: Task) -> float:
        """计算与任务的距离，使用公式(2)"""
        return math.sqrt((self.x - task.x)**2 + (self.y - task.y)**2)

    def calculate_cost(self, task: Task) -> float:
        """计算工人报酬成本，使用公式(3)"""
        distance = self.calculate_distance(task)
        return distance * self.compensation

    def calculate_benefit(self, task: Task, epsilon: float = 1.0) -> float:
        """计算工人收益，使用公式(4)"""
        distance = self.calculate_distance(task)
        travel_time = distance / self.speed
        completion_rate = task.get_completion_rate(travel_time, epsilon)
        return completion_rate * self.competence * task.base_reward * task.priority

class UAV:
    """无人机模型"""
    def __init__(self, uav_id: int, x: float, y: float, energy_consumption: float,
                 competence: float, speed: float):
        self.uav_id = uav_id
        self.x = x  # X
        self.y = y  # Y
        self.energy_consumption = energy_consumption  # B_U_m (每km)
        self.competence = competence  # C_U_m (0.9-1.0)
        self.speed = speed  # V_U_m (9-10)
        self.available = True

    def calculate_distance(self, task: Task) -> float:
        """计算与任务的距离，使用公式(6)"""
        return math.sqrt((self.x - task.x)**2 + (self.y - task.y)**2)

    def calculate_cost(self, task: Task) -> float:
        """计算无人机能耗成本，使用公式(7)"""
        distance = self.calculate_distance(task)
        return distance * self.energy_consumption

    def calculate_benefit(self, task: Task, epsilon: float = 1.0) -> float:
        """使用公式(8)计算无人机收益"""
        distance = self.calculate_distance(task)
        travel_time = distance / self.speed
        completion_rate = task.get_completion_rate(travel_time, epsilon)
        return completion_rate * self.competence * task.base_reward * task.priority

In [3]:
# 查看构建的模型信息
task = Task(task_id=1, x=10, y=20, priority=0.8, base_reward=100, deadline=50)
worker = Worker(worker_id=1, x=5, y=5, compensation=2, competence=0.85, speed=4.5)
uav = UAV(uav_id=1, x=15, y=15, energy_consumption=1.5, competence=0.95, speed=9.5)

print(f"任务完成率: {task.get_completion_rate(30)}")
print(f"工人距任务的距离: {worker.calculate_distance(task)}")
print(f"工人的花费: {worker.calculate_cost(task)}")
print(f"工人的收益: {worker.calculate_benefit(task)}")
print(f"UAV到任务的距离: {uav.calculate_distance(task)}")
print(f"UAV的花费: {uav.calculate_cost(task)}")
print(f"UAV的收益: {uav.calculate_benefit(task)}")


任务完成率: 0.9999999979388463
工人距任务的距离: 15.811388300841896
工人的花费: 31.622776601683793
工人的收益: 68.0
UAV到任务的距离: 7.0710678118654755
UAV的花费: 10.606601717798213
UAV的收益: 76.0
